In [2]:
import numpy as np
import orbitize
from orbitize import read_input, system, priors
import matplotlib.pyplot as plt

In [3]:
data_table = read_input.read_file('{}/betaPic.csv'.format(orbitize.DATADIR))
data_table = data_table[:-1] # Discard the RV observation, as we do not take it into account in the model

num_planets = 1
total_mass = 1.75 # [Msol]
plx = 51.44 # [mas]
mass_err = 0.05 # [Msol]
plx_err = 0.12 # [mas]

sys = system.System(
    num_planets, data_table, total_mass,
    plx, mass_err=mass_err, plx_err=plx_err
)

lab = sys.param_idx



In [4]:
loaded_npe_samples = np.load('/home/ps/4T/npe-astrometry-betapic/flowsamples_2W_12.12.npy')

In [5]:
loaded_npe_samples[0]

array([9.0154352e+00, 1.7706206e-02, 8.8653435e+01, 3.0704947e+00,
       3.2667122e+01, 7.7501094e-01, 5.1345253e+01, 1.7040315e+00],
      dtype=float32)

In [6]:
# set up the same priors as for https://arxiv.org/abs/2201.08506v1(github : https://github.com/HeSunPU/DPI/tree/main)
sys.sys_priors[lab['sma1']] = priors.UniformPrior(loaded_npe_samples[:,0].min(), loaded_npe_samples[:,0].max())
sys.sys_priors[lab['ecc1']] = priors.UniformPrior(0.00001, 2*loaded_npe_samples[:,1].max())
sys.sys_priors[lab['inc1']] = priors.UniformPrior(np.deg2rad(loaded_npe_samples[:,2].min()), np.deg2rad(loaded_npe_samples[:,2].max()))
sys.sys_priors[lab['aop1']] = priors.UniformPrior(loaded_npe_samples[:,3].min(), loaded_npe_samples[:,3].max())
sys.sys_priors[lab['pan1']] = priors.UniformPrior(np.deg2rad(loaded_npe_samples[:,4].min()), np.deg2rad(loaded_npe_samples[:,4].max()))
sys.sys_priors[lab['tau1']] = priors.UniformPrior(loaded_npe_samples[:,5].min(), loaded_npe_samples[:,5].max())
# sys.sys_priors[lab['plx']] = priors.UniformPrior(loaded_npe_samples[:,6].min()*0.95, loaded_npe_samples[:,6].max()*1.05)
sys.sys_priors[lab['mtot']] = priors.UniformPrior(loaded_npe_samples[:,7].min(), loaded_npe_samples[:,7].max())

# sys.sys_priors[lab['sma1']] = priors.UniformPrior(4.0, 40.0)
# sys.sys_priors[lab['ecc1']] = priors.UniformPrior(0.00001, 0.99)
# sys.sys_priors[lab['inc1']] = priors.UniformPrior(np.deg2rad(81), np.deg2rad(99))
# sys.sys_priors[lab['aop1']] = priors.UniformPrior(0, 2*np.pi)
# sys.sys_priors[lab['pan1']] = priors.UniformPrior(np.deg2rad(25), np.deg2rad(85))
# sys.sys_priors[lab['tau1']] = priors.UniformPrior(0, 1)
# number of temperatures & walkers for MCMC
num_temps = 20
num_walkers = 1000

# number of steps to take
n_orbs = 1000000

In [7]:
import numpy as np
import astropy.units as u
import astropy.constants as consts
import abc
import time
from astropy.time import Time

import dynesty

import emcee
import ptemcee
import multiprocessing as mp

from multiprocessing import Pool

import orbitize.lnlike
import orbitize.priors
import orbitize.kepler
from orbitize import cuda_ext

import orbitize.results
import matplotlib.pyplot as plt

In [8]:
class Sampler(abc.ABC):
    """
    Abstract base class for sampler objects.
    All sampler objects should inherit from this class.

    Written: Sarah Blunt, 2018
    """

    def __init__(
        self, system, like="chi2_lnlike", custom_lnlike=None, chi2_type="standard"
    ):
        self.system = system

        # check if `like` is a string or a function
        if callable(like):
            self.lnlike = like
        else:
            self.lnlike = getattr(orbitize.lnlike, like)

        self.custom_lnlike = custom_lnlike
        self.chi2_type = chi2_type
        # check if need to handle covariances
        self.has_corr = np.any(~np.isnan(self.system.data_table["quant12_corr"]))

    @abc.abstractmethod
    def run_sampler(self, total_orbits):
        pass

    def _logl(self, params):
        """
        log likelihood function that interfaces with the orbitize objects
        Comptues the sum of the log likelihoods of the data given the input model

        Args:
            params (np.array of float): RxM array
                of fitting parameters, where R is the number of
                parameters being fit, and M is the number of orbits
                we need model predictions for. Must be in the same order
                documented in System() above. If M=1, this can be a 1d array.

        Returns:
            float: sum of all log likelihoods of the data given input model

        """
        # compute the model based on system params
        model, jitter = self.system.compute_model(params)

        # fold data/errors to match model output shape. In particualr, quant1/quant2 are interleaved
        data = np.array(
            [self.system.data_table["quant1"], self.system.data_table["quant2"]]
        ).T

        # errors below required for lnlike function below
        errs = np.array(
            [self.system.data_table["quant1_err"], self.system.data_table["quant2_err"]]
        ).T
        # covariances/correlations, if applicable
        # we're doing this check now because the likelihood computation is much faster if we can skip it.
        if self.has_corr:
            corrs = self.system.data_table["quant12_corr"]
        else:
            corrs = None

        # grab all seppa indices
        seppa_indices = self.system.all_seppa

        # compute lnlike
        lnlikes = self.lnlike(
            data, errs, corrs, model, jitter, seppa_indices, chi2_type=self.chi2_type
        )

        # return sum of lnlikes (aka product of likeliehoods)
        lnlikes_sum = np.nansum(lnlikes, axis=(0, 1))

        if self.custom_lnlike is not None:
            lnlikes_sum += self.custom_lnlike(params)

        if self.system.hipparcos_IAD is not None:
            # compute Ra/Dec predictions at the Hipparcos IAD epochs
            raoff_model, deoff_model, _ = self.system.compute_all_orbits(
                params, epochs=self.system.hipparcos_IAD.epochs_mjd
            )

            (
                raoff_model_hip_epoch,
                deoff_model_hip_epoch,
                _,
            ) = self.system.compute_all_orbits(
                params, epochs=Time([1991.25], format="decimalyear").mjd
            )

            # subtract off position of star at reference Hipparcos epoch
            raoff_model[:, 0, :] -= raoff_model_hip_epoch[:, 0, :]
            deoff_model[:, 0, :] -= deoff_model_hip_epoch[:, 0, :]

            # select body 0 raoff/deoff predictions & feed into Hip IAD lnlike fn
            lnlikes_sum += self.system.hipparcos_IAD.compute_lnlike(
                raoff_model[:, 0, :],
                deoff_model[:, 0, :],
                params,
                self.system.param_idx,
            )

        if self.system.gaia is not None:
            gaiahip_epochs = Time(
                np.append(
                    self.system.gaia.hipparcos_epoch, self.system.gaia.gaia_epoch
                ),
                format="decimalyear",
            ).mjd

            # compute Ra/Dec predictions at the Gaia epoch
            raoff_model, deoff_model, _ = self.system.compute_all_orbits(
                params, epochs=gaiahip_epochs
            )

            # select body 0 raoff/deoff predictions & feed into Gaia module lnlike fn
            lnlikes_sum += self.system.gaia.compute_lnlike(
                raoff_model[:, 0, :],
                deoff_model[:, 0, :],
                params,
                self.system.param_idx,
            )
        return lnlikes_sum

In [9]:
class MCMC(Sampler):
    """
    MCMC sampler. Supports either parallel tempering or just regular MCMC. Parallel tempering will be run if ``num_temps`` > 1
    Parallel-Tempered MCMC Sampler uses ptemcee, a fork of the emcee Affine-infariant sampler
    Affine-Invariant Ensemble MCMC Sampler uses emcee.

    .. Warning:: may not work well for multi-modal distributions

    Args:
        system (system.System): system.System object
        num_temps (int): number of temperatures to run the sampler at.
            Parallel tempering will be used if num_temps > 1 (default=20)
        num_walkers (int): number of walkers at each temperature (default=1000)
        num_threads (int): number of threads to use for parallelization (default=1)
        chi2_type (str, optional): either  "standard", or "log"
        like (str): name of likelihood function in ``lnlike.py``
        custom_lnlike (func): ability to include an addition custom likelihood
            function in the fit. The function looks like
            ``clnlikes = custon_lnlike(params)`` where ``params`` is a RxM array
            of fitting parameters, where R is the number of orbital paramters
            (can be passed in system.compute_model()), and M is the number of
            orbits we need model predictions for. It returns ``clnlikes``
            which is an array of length M, or it can be a single float if M = 1.
        prev_result_filename (str): if passed a filename to an HDF5 file
            containing a orbitize.Result data, MCMC will restart from where it
            left off.

    Written: Jason Wang, Henry Ngo, 2018
    """

    def __init__(
        self,
        system,
        num_temps=20,
        num_walkers=1000,
        num_threads=1,
        chi2_type="standard",
        like="chi2_lnlike",
        custom_lnlike=None,
        prev_result_filename=None,
        flow=True,
        flow_path=None,
    ):
        super(MCMC, self).__init__(
            system, like=like, chi2_type=chi2_type, custom_lnlike=custom_lnlike
        )
        self.flow = flow
        self.flow_path = flow_path
        self.num_temps = num_temps
        self.num_walkers = num_walkers
        self.num_threads = num_threads

        # create an empty results object
        self.results = orbitize.results.Results(
            self.system,
            sampler_name=self.__class__.__name__,
            post=None,
            lnlike=None,
            version_number=orbitize.__version__,
        )

        if self.num_temps > 1:
            self.use_pt = True
        else:
            self.use_pt = False
            self.num_temps = 1

        # get priors from the system class. need to remove and record fixed priors
        self.priors = []
        self.fixed_params = []

        self.sampled_param_idx = {}
        sampled_param_counter = 0
        for i, prior in enumerate(system.sys_priors):
            # check for fixed parameters
            if not hasattr(prior, "draw_samples"):
                self.fixed_params.append((i, prior))
            else:
                self.priors.append(prior)
                self.sampled_param_idx[self.system.labels[i]] = sampled_param_counter
                sampled_param_counter += 1

        # initialize walkers initial postions
        self.num_params = len(self.priors)
        i = 0
        if prev_result_filename is None:
            # initialize walkers initial postions
            init_positions = []
            for prior in self.priors:
                if self.flow is not None and i == 0:
                    self.flow_path = self.flow_path
                    loaded_npe_samples = np.load(self.flow_path)
                    random_init = loaded_npe_samples[: ,i].astype(np.float64)
                    i += 1
                else:
                    random_init = prior.draw_samples(num_walkers * num_temps)
                if self.num_temps > 1:
                    random_init = random_init.reshape([self.num_temps, num_walkers])

                init_positions.append(random_init)



            # save this as the current position for the walkers
            if self.use_pt:
                # make this an numpy array, but combine the parameters into a shape of (ntemps, nwalkers, nparams)
                # we currently have a list of [ntemps, nwalkers] with nparam arrays. We need to make nparams the third dimension
                self.curr_pos = np.dstack(init_positions)
            else:
                # make this an numpy array, but combine the parameters into a shape of (nwalkers, nparams)
                # we currently have a list of arrays where each entry is num_walkers prior draws for each parameter
                # We need to make nparams the second dimension, so we have to transpose the stacked array
                self.curr_pos = np.stack(init_positions).T
        else:
            # restart from previous walker positions
            self.results.load_results(prev_result_filename, append=True)

            prev_pos = self.results.curr_pos

            # check previous positions has the correct dimensions as we need given how this sampler was created.
            expected_shape = (self.num_walkers, len(self.priors))
            if self.use_pt:
                expected_shape = (self.num_temps,) + expected_shape
            if prev_pos.shape != expected_shape:
                raise ValueError(
                    "Unable to restart chain. Saved walker positions has shape {0}, while current sampler needs {1}".format(
                        prev_pos.shape, expected_shape
                    )
                )

            self.curr_pos = prev_pos

    def _fill_in_fixed_params(self, sampled_params):
        """
        Fills in the missing parameters from the chain that aren't being sampled

        Args:
            sampled_params (np.array): either 1-D array of size = number of
                sampled params, or 2-D array of shape (num_models, num_params)

        Returns:
            np.array: same number of dimensions as sampled_params,
                but with num_params including the fixed parameters
        """
        if len(self.fixed_params) == 0:
            # nothing to add
            return sampled_params

        # check if 1-D or 2-D
        twodim = np.ndim(sampled_params) == 2

        # insert in params
        for index, value in self.fixed_params:
            if twodim:
                sampled_params = np.insert(sampled_params, index, value, axis=1)
            else:
                sampled_params = np.insert(sampled_params, index, value)

        return sampled_params

    def _logl(self, params, include_logp=False):
        """
        log likelihood function that interfaces with the orbitize objects
        Comptues the sum of the log likelihoods of the data given the input model

        Args:
            params (np.array of float): MxR array
                of fitting parameters, where R is the number of
                parameters being fit, and M is the number of orbits
                we need model predictions for. Must be in the same order
                documented in System() above. If M=1, this can be a 1d array.
            include_logp (bool): if True, also include log prior in this function

        Returns:
            lnlikes (float): sum of all log likelihoods of the data given input model

        """
        if include_logp:
            if np.ndim(params) == 1:
                logp = orbitize.priors.all_lnpriors(params, self.priors)
                # escape if logp == -np.inf
                if np.isinf(logp):
                    return -np.inf
            else:
                logp = np.array(
                    [orbitize.priors.all_lnpriors(pset, self.priors) for pset in params]
                )
        else:
            logp = 0  # don't include prior

        full_params = self._fill_in_fixed_params(params)
        if np.ndim(full_params) == 2:
            full_params = full_params.T

        return super(MCMC, self)._logl(full_params) + logp

    def _update_chains_from_sampler(self, sampler, num_steps=None):
        """
        Updates self.post, self.chain, and self.lnlike from the MCMC sampler

        Args:
            sampler (emcee.EnsembleSampler or ptemcee.Sampler): sampler object.
            num_steps (int): if not None, only stores the first num_steps number of steps
        """
        if num_steps is None:
            # use all the steps, grab total number of steps from dimension of chains
            num_steps = sampler.chain.shape[-2]

        self.chain = sampler.chain
        num_params = self.chain.shape[-1]

        if self.use_pt:
            # chain is shape: Ntemp x Nwalkers x Nsteps x Nparams
            self.post = sampler.chain[0, :, :num_steps].reshape(
                -1, num_params
            )  # the reshaping flattens the chain
            # should also be picking out the lowest temperature logps
            self.lnlikes = sampler.loglikelihood[0, :, :num_steps].flatten()
            self.lnlikes_alltemps = sampler.loglikelihood[:, :, :num_steps]
        else:
            # chain is shape: Nwalkers x Nsteps x Nparams
            self.post = sampler.chain[:, :num_steps].reshape(-1, num_params)
            self.lnlikes = sampler.lnprobability[:, :num_steps].flatten()

            # convert posterior probability (returned by sampler objects) to likelihood (required by orbitize.results.Results)
            for i, orb in enumerate(self.post):
                self.lnlikes[i] -= orbitize.priors.all_lnpriors(orb, self.priors)

        # include fixed parameters in posterior
        self.post = self._fill_in_fixed_params(self.post)

    def validate_xyz_positions(self):
        """
        If using the XYZ basis, walkers might be initialized in an invalid
        region of parameter space. This function fixes that by replacing invalid
        positions by new randomly generated positions until all are valid.
        """
        if self.system.fitting_basis == "XYZ":
            if self.use_pt:
                all_valid = False
                while not all_valid:
                    total_invalids = 0
                    for temp in range(self.num_temps):
                        to_stand = self.system.basis.to_standard_basis(
                            self.curr_pos[temp, :, :].T.copy()
                        ).T

                        # Get invalids by checking ecc values for each companion
                        indices = [
                            ((i * 6) + 1)
                            for i in range(self.system.num_secondary_bodies)
                        ]
                        invalids = np.where(
                            (to_stand[:, indices] < 0.0) | (to_stand[:, indices] >= 1.0)
                        )[0]

                        # Redraw samples for the invalid ones
                        if len(invalids) > 0:
                            newpos = []
                            for prior in self.priors:
                                randompos = prior.draw_samples(len(invalids))
                                newpos.append(randompos)
                            self.curr_pos[temp, invalids, :] = np.stack(newpos).T
                            total_invalids += len(invalids)
                    if total_invalids == 0:
                        all_valid = True
                        print("All walker positions validated.")
            else:
                all_valid = False
                while not all_valid:
                    total_invalids = 0
                    to_stand = self.system.basis.to_standard_basis(
                        self.curr_pos[:, :].T.copy()
                    ).T

                    # Get invalids by checking ecc values for each companion
                    indices = [
                        ((i * 6) + 1) for i in range(self.system.num_secondary_bodies)
                    ]
                    invalids = np.where(
                        (to_stand[:, indices] < 0.0) | (to_stand[:, indices] >= 1.0)
                    )[0]

                    # Redraw saples for the invalid ones
                    if len(invalids) > 0:
                        newpos = []
                        for prior in self.priors:
                            randompos = prior.draw_samples(len(invalids))
                            newpos.append(randompos)
                        self.curr_pos[invalids, :] = np.stack(newpos).T
                        total_invalids += len(invalids)
                    if total_invalids == 0:
                        all_valid = True
                        print("All walker positions validated.")

    def run_sampler(
        self,
        total_orbits,
        burn_steps=0,
        thin=1,
        examine_chains=False,
        output_filename=None,
        periodic_save_freq=None,
    ):
        """
        Runs PT MCMC sampler. Results are stored in ``self.chain`` and ``self.lnlikes``.
        Results also added to ``orbitize.results.Results`` object (``self.results``)

        .. Note:: Can be run multiple times if you want to pause and inspect things.
            Each call will continue from the end state of the last execution.

        Args:
            total_orbits (int): total number of accepted possible
                orbits that are desired. This equals
                ``num_steps_per_walker`` x ``num_walkers``
            burn_steps (int): optional paramter to tell sampler
                to discard certain number of steps at the beginning
            thin (int): factor to thin the steps of each walker
                by to remove correlations in the walker steps
            examine_chains (boolean): Displays plots of walkers at each step by
                running `examine_chains` after `total_orbits` sampled.
            output_filename (str): Optional filepath for where results file can be saved.
            periodic_save_freq (int): Optionally, save the current results into ``output_filename``
                every nth step while running, where n is value passed into this variable.

        Returns:
            ``emcee.sampler`` object: the sampler used to run the MCMC
        """

        if periodic_save_freq is not None and output_filename is None:
            raise ValueError(
                "output_filename must be defined for periodic saving of the chains"
            )
        if periodic_save_freq is not None and not isinstance(periodic_save_freq, int):
            raise TypeError("periodic_save_freq must be an integer")

        nsteps = int(np.ceil(total_orbits / self.num_walkers))
        if nsteps <= 0:
            raise ValueError("Total_orbits must be greater than num_walkers.")

        with Pool(processes=self.num_threads) as pool:
            if self.use_pt:
                sampler = ptemcee.Sampler(
                    self.num_walkers,
                    self.num_params,
                    self._logl,
                    orbitize.priors.all_lnpriors,
                    ntemps=self.num_temps,
                    threads=self.num_threads,
                    logpargs=[
                        self.priors,
                    ],
                )
            else:
                sampler = emcee.EnsembleSampler(
                    self.num_walkers,
                    self.num_params,
                    self._logl,
                    pool=pool,
                    kwargs={"include_logp": True},
                )

            print("Starting Burn in")
            for i, state in enumerate(
                sampler.sample(self.curr_pos, iterations=burn_steps, thin=thin)
            ):
                if self.use_pt:
                    self.curr_pos = state[0]
                else:
                    self.curr_pos = state.coords

                if (i + 1) % 5 == 0:
                    print(
                        str(i + 1)
                        + "/"
                        + str(burn_steps)
                        + " steps of burn-in complete",
                        end="\r",
                    )

                if periodic_save_freq is not None:
                    if (i + 1) % periodic_save_freq == 0:  # we've completed i+1 steps
                        self.results.curr_pos = self.curr_pos
                        self.results.save_results(output_filename)

            sampler.reset()
            print("")
            print("Burn in complete. Sampling posterior now.")

            saved_upto = 0  # keep track of how many steps of this chain we've saved. this is the next index that needs to be saved
            for i, state in enumerate(
                sampler.sample(self.curr_pos, iterations=nsteps, thin=thin)
            ):
                if self.use_pt:
                    self.curr_pos = state[0]
                else:
                    self.curr_pos = state.coords

                # print progress statement
                if (i + 1) % 5 == 0:
                    print(str(i + 1) + "/" + str(nsteps) + " steps completed", end="\r")

                if periodic_save_freq is not None:
                    if (i + 1) % periodic_save_freq == 0:  # we've completed i+1 steps
                        self._update_chains_from_sampler(sampler, num_steps=i + 1)

                        # figure out what is the new chunk of the chain and corresponding lnlikes that have been computed before last save
                        # grab the current posterior and lnlikes and reshape them to have the Nwalkers x Nsteps dimension again
                        post_shape = self.post.shape
                        curr_chain_shape = (
                            self.num_walkers,
                            post_shape[0] // self.num_walkers,
                            post_shape[-1],
                        )
                        curr_chain = self.post.reshape(curr_chain_shape)
                        curr_lnlike_chain = self.lnlikes.reshape(curr_chain_shape[:2])
                        # use the reshaped arrays and find the new steps we computed
                        curr_chunk = curr_chain[:, saved_upto : i + 1]
                        curr_chunk = curr_chunk.reshape(
                            -1, curr_chunk.shape[-1]
                        )  # flatten nwalkers x nsteps dim
                        curr_lnlike_chunk = curr_lnlike_chain[
                            :, saved_upto : i + 1
                        ].flatten()

                        # add this current chunk to the results object (which already has all the previous chunks saved)
                        self.results.add_samples(
                            curr_chunk, curr_lnlike_chunk, curr_pos=self.curr_pos
                        )
                        self.results.save_results(output_filename)
                        saved_upto = i + 1

            print("")
            self._update_chains_from_sampler(sampler)

            if periodic_save_freq is None:
                # need to save everything
                self.results.add_samples(
                    self.post, self.lnlikes, curr_pos=self.curr_pos
                )
            elif saved_upto < nsteps:
                # just need to save the last few
                # same code as above except we just need to grab the last few
                post_shape = self.post.shape
                curr_chain_shape = (
                    self.num_walkers,
                    post_shape[0] // self.num_walkers,
                    post_shape[-1],
                )
                curr_chain = self.post.reshape(curr_chain_shape)
                curr_lnlike_chain = self.lnlikes.reshape(curr_chain_shape[:2])
                curr_chunk = curr_chain[:, saved_upto:]
                curr_chunk = curr_chunk.reshape(
                    -1, curr_chunk.shape[-1]
                )  # flatten nwalkers x nsteps dim
                curr_lnlike_chunk = curr_lnlike_chain[:, saved_upto:].flatten()

                self.results.add_samples(
                    curr_chunk, curr_lnlike_chunk, curr_pos=self.curr_pos
                )

            if output_filename is not None:
                self.results.save_results(output_filename)

            print("Run complete")
        # Close pool
        if examine_chains:
            self.examine_chains()

        return sampler

In [ ]:
mcmc_sampler = MCMC(sys, num_temps, num_walkers, num_threads = 10,flow=True,flow_path='/home/ps/4T/npe-astrometry-betapic/flowsamples_2W_12.12.npy')

abs = mcmc_sampler.run_sampler(10000, burn_steps=400, output_filename='FMMCMC-hdf5')